# Earnings Report Alert System

Backtests how S&P 500 stocks react after earnings beats and misses, then flags recently
released earnings reports with a signal based on that history.

**Run order:** run all cells top to bottom. The notebook writes its results to `data/`,
which the Streamlit dashboard (`earnings_app.py`) reads.

> On Kaggle, turn **Internet** on in the session options before running.
>
> **Not investment advice.** See the Limitations section in the README.

## 0. Setup

In [ ]:
# Install dependencies (already available on most Kaggle images)
!pip install -q yfinance

import json
import os
import time
from datetime import timedelta
from io import StringIO

import numpy as np
import pandas as pd
import requests
import yfinance as yf

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# Get the current list of S&P 500 companies from Wikipedia
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
response = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers=headers)
sp500_table = pd.read_html(StringIO(response.text))[0]
# yfinance uses '-' instead of '.' (e.g. BRK.B -> BRK-B)
ALL_SP500_TICKERS = sp500_table["Symbol"].str.replace(".", "-", regex=False).tolist()
print(f"S&P 500 companies: {len(ALL_SP500_TICKERS)}")


def get_earnings_history(ticker, num_quarters=8, max_retries=2, delay=3):
    """Fetch earnings dates, EPS estimates and reported EPS for one ticker."""
    for attempt in range(max_retries):
        try:
            stock = yf.Ticker(ticker)
            earnings = stock.get_earnings_dates(limit=num_quarters)
            if earnings is not None and not earnings.empty:
                earnings = earnings.reset_index()
                earnings["ticker"] = ticker
                return earnings
        except Exception:
            pass
        time.sleep(delay)
    return pd.DataFrame()

## 1. Fetch earnings history for all S&P 500 companies

Takes about 8-10 minutes.

In [ ]:
all_earnings, failed = [], []
for i, ticker in enumerate(ALL_SP500_TICKERS, 1):
    df = get_earnings_history(ticker)
    time.sleep(0.5)  # avoid yfinance rate limiting
    if df.empty:
        failed.append(ticker)
        continue
    # Make the date column name consistent regardless of yfinance version
    df = df.rename(columns={df.columns[0]: "Earnings Date"})
    all_earnings.append(df)
    if i % 100 == 0:
        print(f"[{i}/{len(ALL_SP500_TICKERS)}] tickers processed")

earnings_df = pd.concat(all_earnings, ignore_index=True)
earnings_df.to_csv(f"{DATA_DIR}/earnings_raw.csv", index=False)

print(f"\nTotal earnings records: {len(earnings_df)}")
print(f"Unique companies: {earnings_df['ticker'].nunique()} of {len(ALL_SP500_TICKERS)}")
print(f"Failed tickers: {failed}")
display(earnings_df.head(10))

## 2. Backtest the price reaction after each report

The base price depends on when the report was announced (New York time):

| Announcement | Base price | Day 1 |
|---|---|---|
| Before 16:00 (pre-market or during session) | Previous trading day's close | Announcement day |
| 16:00 or later (after close) | Announcement day's close | Next trading day |

Price history is fetched once per ticker (about 500 requests instead of ~11,800). Takes about 10-15 minutes.

In [ ]:
MARKET_CLOSE_HOUR = 16
DAYS_AFTER = [1, 3, 5]

# Keep only reports already released, and convert dates to New York time
earnings_actual_df = earnings_df.dropna(subset=["Reported EPS", "Surprise(%)"]).copy()
earnings_actual_df["Earnings Date"] = (
    pd.to_datetime(earnings_actual_df["Earnings Date"], utc=True).dt.tz_convert("America/New_York")
)
print(f"Released reports to backtest: {len(earnings_actual_df)}")


def get_price_history(ticker, start, end, max_retries=2, delay=3):
    """Fetch daily closing prices for a ticker, indexed by naive trading date."""
    for _ in range(max_retries):
        try:
            hist = yf.Ticker(ticker).history(start=start, end=end)
            if not hist.empty:
                hist.index = hist.index.tz_localize(None).normalize()
                return hist["Close"]
        except Exception:
            pass
        time.sleep(delay)
    return pd.Series(dtype=float)


def compute_reaction(closes, event_ts):
    """Compute % price change 1/3/5 trading days after an earnings announcement."""
    event_day = event_ts.tz_localize(None).normalize()
    before_close = event_ts.hour < MARKET_CLOSE_HOUR

    if before_close:
        base_prices = closes[closes.index < event_day]
        after_prices = closes[closes.index >= event_day]
    else:
        base_prices = closes[closes.index <= event_day]
        after_prices = closes[closes.index > event_day]

    if base_prices.empty:
        return None

    base_price = base_prices.iloc[-1]
    result = {
        "base_price": round(base_price, 2),
        "timing": "before_close" if before_close else "after_close",
    }
    for days in DAYS_AFTER:
        if len(after_prices) >= days:
            pct = (after_prices.iloc[days - 1] - base_price) / base_price * 100
            result[f"pct_change_{days}d"] = round(pct, 2)
        else:
            result[f"pct_change_{days}d"] = None  # not enough trading days yet
    return result


backtest_results, failed = [], []
groups = earnings_actual_df.groupby("ticker")

for i, (ticker, group) in enumerate(groups, 1):
    start = (group["Earnings Date"].min() - timedelta(days=10)).strftime("%Y-%m-%d")
    end = (group["Earnings Date"].max() + timedelta(days=15)).strftime("%Y-%m-%d")
    closes = get_price_history(ticker, start, end)
    time.sleep(0.5)  # avoid yfinance rate limiting

    if closes.empty:
        failed.append(ticker)
        continue

    for _, row in group.iterrows():
        result = compute_reaction(closes, row["Earnings Date"])
        if result:
            result["ticker"] = ticker
            result["earnings_date"] = row["Earnings Date"].tz_localize(None)
            result["surprise_pct"] = row["Surprise(%)"]
            backtest_results.append(result)

    if i % 100 == 0:
        print(f"[{i}/{groups.ngroups}] tickers processed")

earnings_backtest_df = pd.DataFrame(backtest_results)
earnings_backtest_df["category"] = np.where(earnings_backtest_df["surprise_pct"] > 0, "Beat", "Miss")

print(f"\nBacktested reports: {len(earnings_backtest_df)} of {len(earnings_actual_df)}")
print(f"Failed tickers: {failed}")
print("\nAnnouncement timing:")
print(earnings_backtest_df["timing"].value_counts())

# Reports with a 00:00 timestamp have an unknown announcement time (treated as before close)
unknown_time = (pd.to_datetime(earnings_backtest_df["earnings_date"]).dt.hour == 0).sum()
print(f"Reports with unknown time (00:00): {unknown_time}")

## 3. Beat vs. miss: how did prices react?

In [ ]:
for days in DAYS_AFTER:
    col = f"pct_change_{days}d"
    print(f"\nAfter {days} trading day(s):")
    summary = earnings_backtest_df.groupby("category")[col].agg(
        mean_change="mean",
        median_change="median",
        pct_up=lambda x: (x.dropna() > 0).mean() * 100,
        count="count",
    ).round(2)
    display(summary)

# Does a bigger surprise lead to a bigger move?
print("\nCorrelation between surprise size and price change:")
for days in DAYS_AFTER:
    col = f"pct_change_{days}d"
    valid = earnings_backtest_df.dropna(subset=[col, "surprise_pct"])
    print(f"  {days} day(s): {valid['surprise_pct'].corr(valid[col]):.3f}  (n = {len(valid)})")

## 4. Historical stats and the advice rule

In [ ]:
def compute_backtest_stats(backtest_df, days=5):
    """Compute historical success rate and average change per category."""
    col = f"pct_change_{days}d"
    valid = backtest_df.dropna(subset=[col, "surprise_pct"])
    stats = {}
    for name, mask in [("beat", valid["surprise_pct"] > 0), ("miss", valid["surprise_pct"] <= 0)]:
        subset = valid.loc[mask, col]
        stats[name] = {
            "success_rate_5d": round(float((subset > 0).mean() * 100), 2),
            "avg_change_5d": round(float(subset.mean()), 2),
            "sample_size": int(len(subset)),
        }
    return stats


BACKTEST_STATS = compute_backtest_stats(earnings_backtest_df)
print("Backtest stats (5 trading days):")
for category, stats in BACKTEST_STATS.items():
    print(f"  {category.upper()}: {stats}")


def generate_earnings_advice(report_row: pd.Series) -> dict:
    """
    Generate a BUY / HOLD / SELL signal based on:
    1. Whether the company beat or missed the EPS estimate
    2. The historical success rate of that category in the backtest
    Reports with no EPS estimate are labelled NO ESTIMATE instead of being
    misclassified as a miss.
    """
    surprise = report_row["surprise_pct"]

    # Handle missing surprise (no analyst estimate available)
    if pd.isna(surprise):
        return {
            "ticker": report_row["ticker"],
            "surprise_pct": None,
            "category": "No estimate",
            "decision": "NO ESTIMATE",
            "historical_success_rate": None,
            "historical_avg_change": None,
            "sample_size": None,
            "note": "No EPS estimate available, so the surprise cannot be calculated",
        }

    # Same rule used in the backtest: > 0 is a beat, otherwise a miss
    category = "beat" if surprise > 0 else "miss"
    stats = BACKTEST_STATS[category]

    if category == "beat" and stats["success_rate_5d"] >= 55:
        decision = "BUY"
        note = f"Historically, {stats['success_rate_5d']:.1f}% of beats rose within 5 days"
    elif category == "miss" and stats["success_rate_5d"] <= 45:
        decision = "SELL"
        note = f"Historically, only {stats['success_rate_5d']:.1f}% of misses rose within 5 days"
    else:
        decision = "HOLD"
        note = "Historical signal is not strong enough"

    return {
        "ticker": report_row["ticker"],
        "surprise_pct": surprise,
        "category": "Beat" if category == "beat" else "Miss",
        "decision": decision,
        "historical_success_rate": stats["success_rate_5d"],
        "historical_avg_change": stats["avg_change_5d"],
        "sample_size": stats["sample_size"],
        "note": note,
    }

## 5. Alert: earnings reports released in the last 3 days

Takes about 8-10 minutes.

In [ ]:
def check_fresh_earnings(tickers, freshness_days=3, delay=0.5):
    """
    Check each ticker for an earnings report released within the last
    `freshness_days` days that already has a reported EPS.
    """
    today = pd.Timestamp.now(tz="UTC").tz_localize(None)
    fresh_reports, failed = [], []

    for ticker in tickers:
        earnings = get_earnings_history(ticker, num_quarters=4)
        time.sleep(delay)  # avoid yfinance rate limiting
        if earnings.empty:
            failed.append(ticker)
            continue

        # After reset_index, the date column is always the first column
        date_col = earnings.columns[0]
        earnings[date_col] = pd.to_datetime(earnings[date_col], utc=True).dt.tz_localize(None)

        # Keep only reports already released (Reported EPS available), newest first
        recent = earnings.dropna(subset=["Reported EPS"]).sort_values(date_col, ascending=False)
        if recent.empty:
            continue

        latest = recent.iloc[0]
        days_since = (today - latest[date_col]).days
        if 0 <= days_since <= freshness_days:
            fresh_reports.append({
                "ticker": ticker,
                "earnings_date": latest[date_col],
                "days_since": days_since,
                "eps_estimate": latest["EPS Estimate"],
                "eps_reported": latest["Reported EPS"],
                "surprise_pct": latest["Surprise(%)"],
            })

    print(f"Failed to fetch data for: {len(failed)} of {len(tickers)} tickers")
    if failed:
        print("Failed tickers:", failed)
    return pd.DataFrame(
        fresh_reports,
        columns=["ticker", "earnings_date", "days_since", "eps_estimate", "eps_reported", "surprise_pct"],
    )


fresh_earnings_df = check_fresh_earnings(ALL_SP500_TICKERS, freshness_days=3)
print(f"Recent earnings reports (last 3 days): {len(fresh_earnings_df)}")

advice_df = pd.DataFrame(
    [generate_earnings_advice(row) for _, row in fresh_earnings_df.iterrows()],
    columns=["ticker", "surprise_pct", "category", "decision", "historical_success_rate",
             "historical_avg_change", "sample_size", "note"],
)
display(advice_df)

## 6. Save the data files used by the dashboard

In [ ]:
# 1) Full backtest results
earnings_backtest_df.to_csv(f"{DATA_DIR}/full_backtest.csv", index=False)

# 2) Summary stats (plain Python types so JSON can store them)
summary_stats = {
    "beat_success_rate_5d": BACKTEST_STATS["beat"]["success_rate_5d"],
    "beat_avg_change_5d": BACKTEST_STATS["beat"]["avg_change_5d"],
    "beat_sample_size": BACKTEST_STATS["beat"]["sample_size"],
    "miss_success_rate_5d": BACKTEST_STATS["miss"]["success_rate_5d"],
    "miss_avg_change_5d": BACKTEST_STATS["miss"]["avg_change_5d"],
    "miss_sample_size": BACKTEST_STATS["miss"]["sample_size"],
    "total_companies": int(earnings_backtest_df["ticker"].nunique()),
    "total_reports": int(len(earnings_backtest_df)),
    "last_updated": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d"),
}
with open(f"{DATA_DIR}/summary_stats.json", "w") as f:
    json.dump(summary_stats, f, indent=2)

# 3) Fresh earnings reports with their advice
fresh_with_advice = fresh_earnings_df.merge(
    advice_df[["ticker", "decision", "note"]], on="ticker", how="left"
)
fresh_with_advice.to_csv(f"{DATA_DIR}/fresh_earnings.csv", index=False)

# 4) Average 5-day reaction per company (only companies with 3+ reports)
company_performance = (
    earnings_backtest_df.groupby("ticker")["pct_change_5d"]
    .agg(avg_reaction_5d="mean", num_reports="count")
    .round(2)
    .reset_index()
)
company_performance = company_performance[company_performance["num_reports"] >= 3]
company_performance.to_csv(f"{DATA_DIR}/company_performance.csv", index=False)

print("Saved files:")
for name in sorted(os.listdir(DATA_DIR)):
    size_kb = os.path.getsize(f"{DATA_DIR}/{name}") / 1024
    print(f"  - {name} ({size_kb:.1f} KB)")
print("\nSummary stats:")
print(json.dumps(summary_stats, indent=2))

## 7. (Optional) Test the dashboard without a browser

Checks that `earnings_app.py` runs without errors using Streamlit's built-in test tool.

In [ ]:
!pip install -q streamlit

from streamlit.testing.v1 import AppTest

# AppTest resolves relative paths from the notebook's temp folder, so use the absolute path
app_path = os.path.abspath("earnings_app.py")
at = AppTest.from_file(app_path, default_timeout=60).run()

print("Exceptions:", [e.value for e in at.exception] or "none")
print("Headers:", [h.value for h in at.header])
print("Metrics:", [(m.label, m.value) for m in at.metric])